# Baby Step 9 — Continuous Intelligence and Targeted Rebalancing

This transparent notebook ingests a synthetic environment refresh and one new company, identifies stale claims, recalculates only the affected opportunity set and writes no external action.


## Control question

Can the exo-brain change its internal priority map when the environment changes without erasing the baseline or silently promoting stale evidence?


In [ ]:
from pathlib import Path
import os, json
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    display=print
VAULT_NAME="Alejandro-Reynoso-Investment-Banking-Vault"
env=os.environ.get("VAULT_PATH")
candidates=([Path(env)] if env else [])+[Path.cwd()/VAULT_NAME,Path("/content/drive/MyDrive")/VAULT_NAME,Path("/workspace/scratch/9ba1ff46ede5")/VAULT_NAME]
VAULT=next((p for p in candidates if p.exists()),None)
if VAULT is None: raise FileNotFoundError("Set VAULT_PATH or mount the vault.")
print("Vault:",VAULT)


## 1. Load and validate inputs


In [ ]:
paths={n:VAULT/"Data"/f for n,f in {"companies":"company_master.csv","events":"loop_009_ingestion_events.csv","intake":"loop_009_company_ingestion.csv","stale":"loop_009_stale_claims.csv","rebalance":"loop_009_opportunity_rebalance.csv","checks":"loop_009_monitoring_checks.csv","claims":"claim_register.csv","sources":"source_registry.csv"}.items()}
missing=[str(p) for p in paths.values() if not p.exists()]
assert not missing,missing
data={k:pd.read_csv(v) for k,v in paths.items()}
assert len(data["companies"])==103 and len(data["events"])==5
print({k:len(v) for k,v in data.items()})


## 2. Inspect the five incoming events


In [ ]:
events=data["events"]
assert events["event_id"].is_unique
display(events[["event_id","event_type","subject","prior_state","new_state"]])


## 3. Validate company 103 against the master schema


In [ ]:
companies=data["companies"]; intake=data["intake"]
arctic=companies.query('id=="SYN-103"')
assert len(arctic)==1 and companies["id"].is_unique
assert intake.iloc[0]["intake_status"]=="PASS"
assert int(intake.iloc[0]["schema_fields"])==len(companies.columns)==40
display(arctic[["id","name","revenue_usd_m","revenue_growth_pct","ebitda_usd_m","recurring_revenue_pct"]])


## 4. Detect and restrict stale claims


In [ ]:
stale=data["stale"]; claims=data["claims"]
assert set(stale["claim_id"])=={"CLM-005","CLM-013"}
registered=claims.set_index("id").loc[["CLM-005","CLM-013"]]
assert registered["stale"].astype(str).str.lower().eq("true").all()
display(stale)


## 5. Preserve the baseline and inspect refreshed scores


In [ ]:
reb=data["rebalance"].copy()
assert {"prior_score","refreshed_score","change"}.issubset(reb.columns)
assert (reb["refreshed_score"]-reb["prior_score"]==reb["change"]).all()
ranked=reb.sort_values(["refreshed_score","company"],ascending=[False,True]).reset_index(drop=True)
ranked["rank"]=ranked.index+1
display(ranked[["rank","company","mandate","prior_score","refreshed_score","change"]])


## 6. Confirm the new entrant and affected-subgraph boundary


In [ ]:
arc=ranked.query('company=="ArcticFlow Technologies"').iloc[0]
assert int(arc["rank"])==3 and int(arc["refreshed_score"])==94
assert len(reb)==10
assert len(companies)==103
print("ArcticFlow rank:",int(arc["rank"]),"Affected opportunities:",len(reb),"Universe:",len(companies))


## 7. Visualize the score change


In [ ]:
plot=reb.set_index("company")[["prior_score","refreshed_score"]].sort_values("refreshed_score")
ax=plot.plot(kind="barh",figsize=(9,6),color=["#AAB4BE","#1F6E8C"],title="Affected opportunity scores: baseline vs refreshed")
ax.spines[["top","right"]].set_visible(False); ax.set_xlabel("Scenario score")


## 8. Verify evidence-register advancement


In [ ]:
sources=data["sources"]
assert len(sources)==18 and len(claims)==36
assert set(["SRC-016","SRC-017","SRC-018"]).issubset(set(sources["id"]))
assert set([f"CLM-{i:03d}" for i in range(31,37)]).issubset(set(claims["id"]))
print("Sources:",len(sources),"Claims:",len(claims),"Stale:",claims["stale"].astype(str).str.lower().eq("true").sum())


## 9. Run the monitoring controls


In [ ]:
checks=data["checks"]
assert len(checks)==8 and (checks["status"]=="PASS").all()
display(checks)


## 10. Decision engine


In [ ]:
decision_checks={"five_events":len(events)==5,"one_company_added":len(arctic)==1,"two_stale_restricted":len(stale)==2,"affected_set_only":len(reb)==10,"baseline_preserved":reb["prior_score"].notna().all(),"human_gate":True,"external_action":False}
assert all(v for k,v in decision_checks.items() if k!="external_action") and not decision_checks["external_action"]
recommendation="ACCEPT REFRESHED INTERNAL BASELINE — NO EXTERNAL ACTION"
print(decision_checks); print(recommendation)


## 11. Baby Step 10 input contract


In [ ]:
contract=pd.DataFrame([["Application","Read-only views over companies, opportunities, claims and decisions"],["Permissions","Explicit read/write/external-action matrix"],["Automation","Dry-run schedule with human gates"],["Integration","Vault adapter and reproducible health check"]],columns=["Layer","Required result"])
display(contract)


## Result

The loop is now dynamic: **event → freshness review → affected subgraph → recalculation → human decision → new baseline**, while historical states remain auditable.
